
# ED Agent Mesh — Safety-Governed Scaffold (Shadow Mode)

Decentralized agent mesh with:
- In-process **Event Bus** (pub/sub)
- File-backed, append-only **Ledger** (blockchain-like)
- **Safety Governor** (smart-contract style) enforcing invariants
- Agents: PerceptionRisk (stub), Critic, DiagnosticsCoordinator, CapacityCoordinator, Actuator, Audit
- Three demo scenarios: major trauma, chest pain (STEMI), pregnant syncope

> Replace `PerceptionRiskAgent.infer()` with your hybrid GRU/MLP when ready.


In [70]:

# Cell 1 — Setup
!pip -q install pyyaml
import os, json, time, uuid, hmac, hashlib, yaml
from datetime import datetime
from dataclasses import dataclass, field
from typing import Callable, Dict, Any, List
print("Setup complete.")


Setup complete.


## Cell 2 — In-Process Event Bus

In [71]:

class EventBus:
    def __init__(self):
        self.subs = {}
    def subscribe(self, topic: str, fn: Callable[[Dict[str, Any]], None]):
        self.subs.setdefault(topic, []).append(fn)
    def publish(self, topic: str, msg: Dict[str, Any]):
        for pattern, fns in self.subs.items():
            if self._match(topic, pattern):
                for fn in fns:
                    fn({"_topic": topic, **msg})
    @staticmethod
    def _match(topic: str, pattern: str) -> bool:
        if pattern.endswith(".*"):
            return topic.startswith(pattern[:-2])
        return topic == pattern

BUS = EventBus()
print("Event bus ready.")


Event bus ready.


## Cell 3 — File-Backed Ledger (Blockchain-like)

In [72]:

LEDGER_PATH = "/mnt/data/ed_agent_mesh_ledger.jsonl"
LEDGER_INDEX = []

def _hash(data: dict) -> str:
    s = json.dumps(data, sort_keys=True, separators=(",",":"))
    return hashlib.sha256(s.encode()).hexdigest()

def ledger_init():
    global LEDGER_INDEX
    LEDGER_INDEX = []
    os.makedirs(os.path.dirname(LEDGER_PATH), exist_ok=True)  # ensure dir exists
    if os.path.exists(LEDGER_PATH):
        os.remove(LEDGER_PATH)
    genesis = {
        "kind": "genesis",
        "ts": datetime.utcnow().isoformat(),
        "prev": None,
        "payload": {"note":"ED Agent Mesh genesis"},
    }
    genesis["hash"] = _hash(genesis)
    with open(LEDGER_PATH, "w") as f:
        f.write(json.dumps(genesis)+"\n")
    LEDGER_INDEX.append(genesis["hash"])

def ledger_append(kind: str, payload: dict) -> dict:
    prev_hash = LEDGER_INDEX[-1] if LEDGER_INDEX else None
    block = {
        "kind": kind,
        "ts": datetime.utcnow().isoformat(),
        "prev": prev_hash,
        "payload": payload,
    }
    block["hash"] = _hash(block)
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(block)+"\n")
    LEDGER_INDEX.append(block["hash"])
    return block

def ledger_tail(n=5):
    if not os.path.exists(LEDGER_PATH):
        return []
    with open(LEDGER_PATH) as f:
        lines = f.readlines()
    return [json.loads(x) for x in lines[-n:]]

ledger_init()
print("Ledger initialized at", LEDGER_PATH)


Ledger initialized at /mnt/data/ed_agent_mesh_ledger.jsonl


## Cell 4 — Policy & Invariants

In [73]:

policy_yaml = '''
version: 0.3
facility: University Hospital ED
approvals:
  red:
    ecg: AUTO
    labs_standard_ed_panel: AUTO
    ct_resus: AUTO
    specialist_page: AUTO
  yellow_green:
    ecg: AUTO
    labs_standard_ed_panel: AUTO
    ct_resus: ATTENDING_CONFIRM
safety_invariants:
  pregnancy_ionizing_imaging:
    rule: "If pregnancy==true and imaging.modality in [CT, XRay] then require attending_override unless life_saving"
  contrast_allergy_renal:
    rule: "If contrast==iodinated and (severe_allergy or eGFR < 30) then require premed or prohibit"
  blood_products_non_mtp:
    rule: "Non-MTP blood requires attending approval"
sla_targets:
  ecg_from_door_minutes: 10
capacity:
  ct_scanners: 2
'''
POLICY = yaml.safe_load(policy_yaml)
print("Invariants:", list(POLICY["safety_invariants"].keys()))


Invariants: ['pregnancy_ionizing_imaging', 'contrast_allergy_renal', 'blood_products_non_mtp']


## Cell 5 — Identity & Signing (Demo HMAC)

In [74]:

SECRET_KEYS = {
    "perception.risk": b"risk-key",
    "critic": b"critic-key",
    "coord.diagnostics": b"diag-key",
    "coord.capacity": b"cap-key",
    "actuator": b"act-key",
    "governor": b"gov-key",
    "audit": b"aud-key",
}
def sign(agent: str, message: dict) -> str:
    s = json.dumps(message, sort_keys=True, separators=(",",":")).encode()
    return hmac.new(SECRET_KEYS[agent], s, hashlib.sha256).hexdigest()
def verify(agent: str, message: dict, signature: str) -> bool:
    return sign(agent, message) == signature
print("Signing ready.")


Signing ready.


In [75]:
# Cell 5.5 — Simple ICU capacity model + helpers

CAPACITY = {
    "ICU": {"total": 2, "occupied": 2},        # start FULL to see blocks
    "CardiologyWard": {"total": 4, "occupied": 1}
}

def capacity_available(kind: str) -> bool:
    pool = CAPACITY.get(kind, {"total": 0, "occupied": 0})
    return pool["occupied"] < pool["total"]

def occupy(kind: str, n: int = 1):
    pool = CAPACITY[kind]
    pool["occupied"] = min(pool["total"], pool["occupied"] + n)

def release(kind: str, n: int = 1):
    pool = CAPACITY[kind]
    pool["occupied"] = max(0, pool["occupied"] - n)

def check_capacity(payload: dict):
    """
    Return (available: bool, reason: str) for a bed request payload.
    """
    level = (payload.get("level") or "Ward").upper()
    if level == "ICU":
        ok = capacity_available("ICU")
        return ok, ("ICU space" if ok else "ICU full")
    # keep wards simple for now
    return True, "Ward assumed available"


In [76]:
# Cell 5.6 — Identity binding registry (EMS <-> MRN)

IDENTITY = {
    # encounter_id -> {"bound": bool, "mrn": Optional[str]}
}

def bind_identity(encounter: str, mrn: str):
    IDENTITY[encounter] = {"bound": True, "mrn": mrn}

def identity_bound(encounter: str) -> bool:
    return IDENTITY.get(encounter, {}).get("bound", False)


In [77]:
# Cell 5.7 — Simulation clock (compressed time)
from datetime import timedelta

SIM_MINUTE_REAL_SECONDS = 0  # keep 0 so advances are instant
_SIM_T0 = datetime.utcnow()
_SIM_OFFSET = timedelta(0)

def sim_now():
    return _SIM_T0 + _SIM_OFFSET

def advance_minutes(n: int):
    """Advance simulated time by n minutes and process pending work."""
    global _SIM_OFFSET
    _SIM_OFFSET += timedelta(minutes=n)
    process_pending()  # defined in the next cell


In [78]:
# Cell 5.8 — Pending work registries
PENDING = {
    "icu_downgrade": {},   # enc -> {"due": ts, "status": "pending", "patient": ..., "note": ...}
    "icu_admit": {},       # enc -> {"due": ts, "status": "pending", "patient": ...}
}

ED_ARRIVAL = {}  # enc -> first-seen time (sim time)

def mark_ed_arrival(encounter: str):
    ED_ARRIVAL.setdefault(encounter, sim_now())

def process_pending():
    """Execute any pending tasks whose due time has passed (uses sim_now)."""
    now = sim_now()

    # 1) ICU downgrades (joker moves)
    for enc, task in list(PENDING["icu_downgrade"].items()):
        if task["status"] == "pending" and task["due"] <= now:
            ok = use_joker()
            ledger_append("action", {"kind": "icu_downgrade_complete", "encounter": enc,
                                     "executed": ok, "ts": now.isoformat()})
            task["status"] = "done"
            # After freeing ICU via joker, re-request ICU bed for this patient
            msg = base_msg("proposal.bed.request", task["patient"], enc,
                           {"service":"Trauma","level":"ICU"}, {}, "fallback")
            BUS.publish("proposal.bed.request", msg)

    # 2) ICU admits (physical transfer)
    for enc, task in list(PENDING["icu_admit"].items()):
        if task["status"] == "pending" and task["due"] <= now:
            ledger_append("action", {"kind": "icu_admit_complete", "encounter": enc,
                                     "executed": True, "ts": now.isoformat()})
            task["status"] = "done"


## Cell 6 — Safety Governor (Smart-Contract Style)

In [79]:
 ## Cell 6 — Safety Governor (identity binding + overrides + capacity checks)

class SafetyGovernor:
    def __init__(self, bus: EventBus, policy: dict):
        self.bus = bus
        self.policy = policy
        bus.subscribe("proposal.*", self.on_proposal)

    def on_proposal(self, msg: dict):
        decision, reason = self.evaluate(msg)
        out = {
            "proposal_id": msg.get("id"),
            "patient": msg.get("patient"),
            "encounter": msg.get("encounter"),
            "verdict": decision,
            "reason": reason,
            "permit_id": None,
            "proposal": msg,  # echo original proposal for actuator use
        }
        if decision == "permit":
            out["permit_id"] = "perm_" + uuid.uuid4().hex[:8]
            ledger_append("permit", {"proposal": msg, "permit": out})
            self.bus.publish("permit.issued", out)
        else:
            ledger_append("block", {"proposal": msg, "block": out})
            self.bus.publish("block.issued", out)

    def evaluate(self, msg: dict):
        p = (msg.get("payload") or {})
        meta = (msg.get("meta") or {})
        typ = msg.get("type", "")

        # 0) Identity binding for ANY order (allow explicit emergency override)
        if typ.startswith("proposal.order."):
            enc = msg.get("encounter")
            if not identity_bound(enc) and not meta.get("identity_override", False):
                return "block", "Identity not bound EMS↔MRN (set meta.identity_override=True if true emergency)"

        # 1) Pregnancy + ionizing imaging: require attending override OR life-saving
        if typ == "proposal.order.ct":
            preg = bool(meta.get("pregnancy_flag", False))
            life_save = bool(meta.get("life_saving", False))
            attending_override = bool(meta.get("attending_override", False))
            if preg and not (life_save or attending_override):
                return "block", "Pregnancy+ionizing CT requires attending override unless life-saving"

        # 2) Contrast allergy / renal function (iodinated contrast)
        if typ == "proposal.order.ct" and p.get("contrast", "none") == "iodinated":
            if meta.get("severe_contrast_allergy", False):
                return "block", "Severe contrast allergy without premed"
            egfr = meta.get("eGFR", 999)
            if egfr is not None and egfr < 30:
                return "block", "eGFR<30: iodinated contrast contraindicated"

        # 3) Non-MTP blood requires attending approval
        if typ == "proposal.order.blood" and not meta.get("mtp", False):
            if not meta.get("attending_approved", False):
                return "block", "Non-MTP blood requires attending approval"

        # 4) Bed capacity gate for ICU
        if typ == "proposal.bed.request":
            level = (p.get("level") or "Ward").upper()
            if level == "ICU":
                ok, _why = check_capacity(p)
                if not ok and not meta.get("attending_override", False):
                    return "block", "ICU full — propose ED hold or transfer"

        # OPTIONAL (nice audit trail): explicitly permit low-risk if identity bound
        if typ in ("proposal.order.ecg", "proposal.order.labs", "proposal.page", "proposal.order.ultrasound"):
            return "permit", "Low-risk order permitted (identity bound)"
       
        # Always permit ED hold proposals (safe default)
        if typ == "proposal.ed_hold":
            return "permit", "ED hold permitted (ICU full / flow safety)"

        # Allow ICU downgrade proposals (joker negotiation decision)
        if typ == "proposal.icu_downgrade":
            return "permit", "ICU downgrade permitted (joker negotiation)"


        # Allow ICU downgrade proposals (joker negotiation decision)
        if typ == "proposal.icu_downgrade":
            return "permit", "ICU downgrade permitted (joker negotiation)"

        return "permit", "All invariants satisfied"

# Re-bind the governor with the updated logic
GOVERNOR = SafetyGovernor(BUS, POLICY)
print("Governor online (identity binding + overrides + capacity).")


Governor online (identity binding + overrides + capacity).


## Cell 7 — Agents

In [80]:
## Cell 7 — Agents (capacity-aware Actuator; ICU requests for major trauma)

def base_msg(_type, patient, encounter, payload=None, meta=None, agent=""):
    msg = {
        "id": "evt_" + uuid.uuid4().hex[:8],
        "type": _type,
        "patient": patient,
        "encounter": encounter,
        "payload": payload or {},
        "meta": meta or {},
        "provenance": {
            "agent": agent,
            "policy": POLICY.get("version"),
            "ts": datetime.utcnow().isoformat()
        }
    }
    return msg

class PerceptionRiskAgent:
    def __init__(self, bus):
        self.bus = bus
        self.adapter = ModelAdapter(device="cpu", weights_path=None, version="hybrid-gru@2025-08-09")

    def infer(self, scenario, features):
        pred = self.adapter.predict(scenario, features)
        s = pred.get("scores", {})
        out = {}
        # fast-lane booleans still there
        if scenario == "major_trauma":
            out["trauma_high"] = s.get("trauma_high", 0.0) >= self.adapter.thresh["trauma_high"]
        elif scenario == "chest_pain":
            out["stemi"] = s.get("stemi", 0.0) >= self.adapter.thresh["stemi"]
        elif scenario == "pregnant_syncope":
            out["late_trimester"] = s.get("late_trimester", 0.0) >= 0.5
            out["fetal_concern"]  = s.get("fetal_concern", 0.0)  >= self.adapter.thresh["fetal_concern"]

        # **new universal care-level signals**
        out["icu_need"] = s.get("icu_need", 0.0) >= self.adapter.thresh["icu_need"]
        out["imc_need"] = s.get("imc_need", 0.0) >= self.adapter.thresh.get("imc_need", 0.5)
        out["deterioration_6h"] = s.get("deterioration_6h", 0.0) >= self.adapter.thresh["deterioration_6h"]
        out["boarding_6h"] = s.get("boarding_6h", 0.0) >= self.adapter.thresh["boarding_6h"]

        # attach raw for audit
        out["_raw"] = s
        out["_model_version"] = pred.get("model_version")
        return out

    def publish(self, patient, encounter, scenario, features):
        mark_ed_arrival(encounter)
        scores = self.infer(scenario, features)
        msg = base_msg("hypothesis."+scenario, patient, encounter, {"scores": scores, "features": features}, agent="perception.risk")
        msg["sig"] = sign("perception.risk", msg)
        BUS.publish("hypothesis."+scenario, msg)


class CriticAgent:
    def __init__(self, bus):
        self.bus = bus
        bus.subscribe("hypothesis.*", self.on_hypothesis)
    def on_hypothesis(self, msg):
        verdict = "agree"
        if msg["_topic"].endswith("chest_pain") and not msg["payload"]["scores"].get("stemi", False):
            verdict = "neutral"
        if msg["_topic"].endswith("pregnant_syncope"):
            s = msg["payload"]["scores"]
            if s.get("late_trimester", False) and s.get("fetal_concern", False):
                verdict = "agree"
        out = base_msg("critic."+msg["_topic"], msg["patient"], msg["encounter"], {"verdict":verdict}, agent="critic")
        out["sig"] = sign("critic", out); BUS.publish("critic."+msg["_topic"], out)

class DiagnosticsCoordinator:
    def __init__(self, bus):
        self.bus = bus
        bus.subscribe("hypothesis.*", self.on_hypothesis)
    def on_hypothesis(self, msg):
        p = msg["payload"]; patient, enc = msg["patient"], msg["encounter"]
        features = p.get("features", {})
        # major trauma → CT WB (life-saving) (still checked by Governor for pregnancy)
        if msg["_topic"].endswith("major_trauma") and p["scores"].get("trauma_high", False):
            meta = {"pregnancy_flag": features.get("pregnancy", False), "life_saving": True}
            m = base_msg("proposal.order.ct", patient, enc, {"study":"CT whole-body","contrast":"iodinated"}, meta, "coord.diagnostics")
            m["sig"] = sign("coord.diagnostics", m); BUS.publish("proposal.order.ct", m)
        # chest pain → labs + ECG (+ page cardiology if STEMI)
        if msg["_topic"].endswith("chest_pain"):
            m1 = base_msg("proposal.order.labs", patient, enc, {"panel":"chest_pain_profile"}, {}, "coord.diagnostics")
            m1["sig"] = sign("coord.diagnostics", m1); BUS.publish("proposal.order.labs", m1)
            m2 = base_msg("proposal.order.ecg", patient, enc, {"repeat":"30min"}, {}, "coord.diagnostics")
            m2["sig"] = sign("coord.diagnostics", m2); BUS.publish("proposal.order.ecg", m2)
            if p["scores"].get("stemi", False):
                m3 = base_msg("proposal.page", patient, enc, {"role":"Cardiology","reason":"STEMI suspected"}, {}, "coord.diagnostics")
                m3["sig"] = sign("coord.diagnostics", m3); BUS.publish("proposal.page", m3)
        # pregnant syncope → OB US if fetal concern
        if msg["_topic"].endswith("pregnant_syncope") and p["scores"].get("fetal_concern", False):
            meta = {"pregnancy_flag": True}
            m = base_msg("proposal.order.ultrasound", patient, enc, {"study":"OB ultrasound"}, meta, "coord.diagnostics")
            m["sig"] = sign("coord.diagnostics", m); BUS.publish("proposal.order.ultrasound", m)

class CapacityCoordinator:
    def __init__(self, bus):
        self.bus = bus
        bus.subscribe("hypothesis.*", self.on_hypothesis)
    def on_hypothesis(self, msg):
        patient, enc = msg["patient"], msg["encounter"]
        # major trauma → request ICU bed
        if msg["_topic"].endswith("major_trauma") and msg["payload"]["scores"].get("trauma_high", False):
            m = base_msg("proposal.bed.request", patient, enc, {"service":"Trauma","level":"ICU"}, {}, "coord.capacity")
            m["sig"] = sign("coord.capacity", m); BUS.publish("proposal.bed.request", m)
        # STEMI → ward bed (keep simple for demo)
        if msg["_topic"].endswith("chest_pain") and msg["payload"]["scores"].get("stemi", False):
            m = base_msg("proposal.bed.request", patient, enc, {"service":"Cardiology","level":"Ward"}, {}, "coord.capacity")
            m["sig"] = sign("coord.capacity", m); BUS.publish("proposal.bed.request", m)

class ActuatorAgent:
    def __init__(self, bus: EventBus):
        self.bus = bus
        bus.subscribe("permit", self.on_permit)

    def on_permit(self, msg):
        prop = msg.get("proposal", {})
        typ = prop.get("type", "")
        enc = prop.get("encounter")
        patient = prop.get("patient")
        now = sim_now()

        if typ == "proposal.icu_downgrade":
            # schedule joker move for 2 simulated hours later
            due = now + timedelta(hours=2)
            PENDING["icu_downgrade"][enc] = {
                "due": due, "status": "pending",
                "patient": patient, "note": "joker move"
            }
            ledger_append("action", {
                "kind": "icu_downgrade_scheduled",
                "encounter": enc, "due": due.isoformat()
            })

        elif typ == "proposal.bed.request" and prop.get("level","").upper() == "ICU":
            # This is a successful ICU admit — schedule physical transfer
            due = now + timedelta(hours=2)
            PENDING["icu_admit"][enc] = {
                "due": due, "status": "pending",
                "patient": patient
            }
            ledger_append("action", {
                "kind": "icu_admit_scheduled",
                "encounter": enc, "due": due.isoformat()
            })

        elif typ == "proposal.ed_hold":
            # Mark arrival for SLA tracking
            mark_ed_arrival(enc)


    def on_permit(self, msg):
        # Execute any permitted action (including ICU bed requests, CT, etc.)
        prop = msg.get("proposal", {})
        if prop.get("type") == "proposal.bed.request":
            lvl = (prop.get("payload", {}).get("level") or "Ward").upper()
            if lvl == "ICU":
                occupy("ICU", 1)
        ledger_append("action", {"permit": msg, "executed": True})

class AuditAgent:
    def __init__(self, bus):
        for topic in ["hypothesis.*","critic.*","proposal.*","permit.*","block.*"]:
            bus.subscribe(topic, self.on_any)
    def on_any(self, msg):
        ledger_append("audit", {"topic": msg["_topic"], "id": msg.get("id"), "ts": datetime.utcnow().isoformat()})

# Re-instantiate agents to rebind subscriptions with the new classes
RISK = PerceptionRiskAgent(BUS)
CRITIC = CriticAgent(BUS)
DIAG = DiagnosticsCoordinator(BUS)
CAP = CapacityCoordinator(BUS)
ACT = ActuatorAgent(BUS)
AUD = AuditAgent(BUS)
print("Agents online (capacity-aware).")


Agents online (capacity-aware).


In [91]:
# Cell 7d — ICUJustificationAgent: propose ICU bed WITH justification when risk says ICU is needed

def _justify_payload(features: dict, scores: dict):
    """
    Build an objective justification bundle commonly used by ICU coordinators.
    `scores` can be the `_raw` model probs if available.
    """
    # fallback NEWS2 if not already in features
    try:
        news2_val = float(features.get("news2", _news2_stub(features)))
    except Exception:
        news2_val = None

    just = {
        "NEWS2": news2_val,
        "SBP": features.get("sbp"),
        "HR": features.get("hr"),
        "RR": features.get("rr"),
        "SpO2": features.get("spo2"),
        "FiO2": features.get("fio2"),
        "Lactate": features.get("lactate"),
        "GCS": features.get("gcs"),
        "Vasopressor": features.get("vasopressor", 0),
        "ModelScores": {},
        "ModelVersion": None,
    }
    # attach raw model scores if present
    if scores:
        raw = scores.get("_raw", {}) if isinstance(scores, dict) else {}
        just["ModelScores"] = {k: float(v) for k, v in raw.items() if isinstance(v, (int, float))}
        just["ModelVersion"] = scores.get("_model_version")
    return just

class ICUJustificationAgent:
    """
    Listens to hypothesis.* messages. If the case meets ICU criteria, it publishes a
    proposal.bed.request with a justification packet in meta.
    """
    def __init__(self, bus, icu_thresh: float = 0.6):
        self.bus = bus
        self.thresh = icu_thresh
        bus.subscribe("hypothesis.*", self.on_hypothesis)

    def on_hypothesis(self, msg: dict):
        payload = msg.get("payload", {}) or {}
        scores  = payload.get("scores", {}) or {}
        feats   = payload.get("features", {}) or {}
        patient, enc = msg["patient"], msg["encounter"]

        # Prefer boolean flag from PerceptionRisk; fall back to raw score threshold
        icu_bool = bool(scores.get("icu_need", False))
        icu_raw  = 0.0
        raw_block = scores.get("_raw", {}) if isinstance(scores, dict) else {}
        if isinstance(raw_block, dict):
            icu_raw = float(raw_block.get("icu_need", 0.0))

        needs_icu = icu_bool or (icu_raw >= self.thresh)
        if not needs_icu:
            return

        meta = {"justification": _justify_payload(feats, scores)}
        # Service name is a placeholder; tweak to local routing rules as needed
        m = base_msg("proposal.bed.request", patient, enc,
                     {"service":"InternalMedicine","level":"ICU"}, meta, agent="coord.capacity")
        m["sig"] = sign("coord.capacity", m)
        self.bus.publish("proposal.bed.request", m)

# Instantiate on the current BUS
ICUJUST = ICUJustificationAgent(BUS, icu_thresh=0.6)  # lower/higher if you want to be stricter/looser
print("ICUJustificationAgent online (adds justification to ICU bed proposals when ICU need is detected).")


ICUJustificationAgent online (adds justification to ICU bed proposals when ICU need is detected).


In [81]:
# Cell 7a — ModelAdapter (extended for ICU/boarding use-cases)
import math
from typing import Optional

def _clipf(v, lo, hi, default):
    try:
        x = float(v)
        if math.isnan(x) or math.isinf(x): return default
        return max(lo, min(hi, x))
    except Exception:
        return default

def _news2_stub(x):
    # Minimal NEWS2-ish proxy: RR, SpO2, SBP, HR, Temp, AVPU ~ GCS
    rr = x.get("rr", 16); spo2 = x.get("spo2", 97); sbp = x.get("sbp", 120)
    hr = x.get("hr", 80); temp = x.get("temp_c", 37); gcs = x.get("gcs", 15)
    score = 0
    # crude bins; replace with proper NEWS2 when you wire real features
    score += 3 if rr>=25 else (2 if rr>=21 else (1 if rr>=12 and rr<=20 else 0))
    score += 3 if spo2<91 else (2 if spo2<93 else (1 if spo2<95 else 0))
    score += 3 if sbp<=90 else (2 if sbp<=100 else 0)
    score += 3 if hr>=130 else (2 if hr>=111 else (1 if hr>=91 else 0))
    score += 1 if temp<36 or temp>=39 else 0
    score += 3 if gcs<15 else 0
    return float(score)

class ModelAdapter:
    def __init__(self, device: str = "cpu", weights_path: Optional[str] = None, version: str = "hybrid-gru@demo"):
        self.device = device
        self.version = version
        self.encoder = None
        self.model = None
        self.ready = False
        try:
            # TODO: load encoder/model here; set self.ready=True when done
            self.ready = (self.encoder is not None and self.model is not None)
        except Exception as e:
            ledger_append("model_error", {"phase":"init", "error": str(e), "version": self.version})
            self.ready = False

        self.thresh = {
            "stemi": 0.80,
            "trauma_high": 0.50,
            "fetal_concern": 0.60,
            "icu_need": 0.60,          # tune with your ROC/PR
            "imc_need": 0.50,
            "deterioration_6h": 0.40,
            "boarding_6h": 0.50,
        }

    def _normalize(self, feats: dict) -> dict:
        mm = dict(feats)
        mm["sbp"] = _clipf(mm.get("sbp"), 50, 250, 120)
        mm["hr"]  = _clipf(mm.get("hr"),  20, 220, 80)
        mm["rr"]  = _clipf(mm.get("rr"),  4,  50, 16)
        mm["spo2"] = _clipf(mm.get("spo2"), 50, 100, 97)
        mm["temp_c"] = _clipf(mm.get("temp_c"), 32, 42, 37)
        mm["gcs"] = _clipf(mm.get("gcs"), 3, 15, 15)
        mm["pregnancy"] = bool(mm.get("pregnancy", False))
        mm["stemi_ecg"] = bool(mm.get("stemi_ecg", False))
        mm["fetal_abn"] = bool(mm.get("fetal_abn", False))
        # optional labs
        for k in ["egfr","lactate","trop_hs","fio2","vasopressor"]:
            if k in mm and mm[k] is not None:
                try: mm[k] = float(mm[k])
                except: mm[k] = None
        # derived
        mm["news2"] = _news2_stub(mm)
        return mm

    def predict(self, scenario: str, feats: dict) -> dict:
        x = self._normalize(feats)

        if not self.ready:
            # ----- deterministic stub focusing on ICU/boarding use-cases -----
            scores = {}
            if scenario == "major_trauma":
                scores["trauma_high"] = 1.0 if (x["sbp"]<90 or x["hr"]>120 or x["gcs"]<=8) else 0.2
                # ICU need proxy from NEWS2 and shock
                scores["icu_need"] = min(1.0, 0.15*x["news2"] + (0.5 if x["sbp"]<90 else 0) + (0.2 if x["gcs"]<13 else 0))
            elif scenario == "chest_pain":
                scores["stemi"] = 0.95 if x["stemi_ecg"] else (0.6 if (x.get("trop_hs") and x["trop_hs"]>50) else 0.1)
                scores["icu_need"] = 0.7 if x["stemi_ecg"] else 0.3
            elif scenario == "pregnant_syncope":
                scores["late_trimester"] = 1.0 if (feats.get("weeks",0)>=28) else 0.2
                scores["fetal_concern"] = 0.8 if x["fetal_abn"] else 0.2
                scores["icu_need"] = 0.4 if x["fetal_abn"] else 0.2
            # generic deterioration/boarding proxies
            scores["deterioration_6h"] = min(1.0, 0.08*x["news2"] + (0.2 if x["spo2"]<92 else 0) + (0.2 if x.get("lactate",0)>3.0 else 0))
            scores["boarding_6h"] = min(1.0, 0.05*x["news2"] + (0.2 if not scores.get("icu_need",0)>0.6 else 0.1))
            return {"model_version": self.version + "+stub", "scores": scores, "uncertainty": {k:0.2 for k in scores}}

        # ===== Real path (fill this when ready) =====
        # with torch.no_grad():
        #     enc = self.encoder.transform(x)
        #     out = self.model(enc)
        # scores = {
        #     "trauma_high": float(out["trauma_prob"]),
        #     "stemi": float(out["stemi_prob"]),
        #     "fetal_concern": float(out["fetal_prob"]),
        #     "icu_need": float(out["icu_prob"]),
        #     "imc_need": float(out["imc_prob"]),
        #     "deterioration_6h": float(out["det6h_prob"]),
        #     "boarding_6h": float(out["board6h_prob"]),
        # }
        # return {"model_version": self.version, "scores": scores, "uncertainty": {k:0.2 for k in scores}}


In [82]:
# Cell 7.5 — Fallback agent with per-encounter dedupe + cooldown

from datetime import timedelta

FALLBACK_STATE = {
    # encounter -> {"last_reason": str, "last_ts": sim_now(), "ed_hold_sent": bool}
}

FALLBACK_COOLDOWN_MIN = 15  # don't re-emit fallback for same reason within 15 simulated minutes

class FallbackAgent:
    def __init__(self, bus):
        self.bus = bus
        bus.subscribe("block.issued", self.on_block)

    def _should_emit(self, encounter: str, reason: str):
        now = sim_now()
        st = FALLBACK_STATE.get(encounter, {})
        last_r = st.get("last_reason")
        last_t = st.get("last_ts")
        if last_r != reason:
            return True
        if last_t is None:
            return True
        # cooldown window
        return (now - last_t) >= timedelta(minutes=FALLBACK_COOLDOWN_MIN)

    def on_block(self, msg):
        prop = msg.get("proposal") or {}
        typ = prop.get("type")
        reason = msg.get("reason","")
        patient, enc = msg.get("patient"), msg.get("encounter")

        # Only react to ICU-full blocks on bed requests
        if typ != "proposal.bed.request" or "ICU full" not in reason:
            return

        # Dedupe + cooldown
        if not self._should_emit(enc, reason):
            return

        # Update state
        FALLBACK_STATE[enc] = {"last_reason": reason, "last_ts": sim_now(),
                               "ed_hold_sent": FALLBACK_STATE.get(enc,{}).get("ed_hold_sent", False)}

        # 1) ED hold once per encounter
        if not FALLBACK_STATE[enc]["ed_hold_sent"]:
            hold = base_msg("proposal.ed_hold", patient, enc,
                            {"reason":"ICU full","service":prop.get("payload",{}).get("service","ED")},
                            {}, "fallback")
            BUS.publish("proposal.ed_hold", hold)
            FALLBACK_STATE[enc]["ed_hold_sent"] = True

        # 2) Try joker first (if available); else prep transfer query
        if joker_available():
            down = base_msg("proposal.icu_downgrade", patient, enc,
                            {"target_ward":"GenWard","free_beds":1}, {}, "fallback")
            BUS.publish("proposal.icu_downgrade", down)
        else:
            transfer = base_msg("proposal.transfer.query", patient, enc,
                                {"level":"ICU","region":"metro"}, {}, "fallback")
            BUS.publish("proposal.transfer.query", transfer)

print("Fallback agent updated: dedupe + 15-min cooldown + ED-hold-once.")


Fallback agent updated: dedupe + 15-min cooldown + ED-hold-once.


In [83]:
ledger_init()
CAPACITY["ICU"]["occupied"] = CAPACITY["ICU"]["total"]  # force contention to exercise fallback
# a “yellow” case with high NEWS2 and lactate but not trauma/STEMI
RISK.publish("tmp-border","ems-border","chest_pain",{
    "stemi_ecg": False, "sbp": 95, "hr": 120, "rr": 26, "spo2": 90, "lactate": 3.5, "fio2": 0.4, "gcs": 15
})
time.sleep(0.2)

# inspect only this encounter
rows = [json.loads(x) for x in open(LEDGER_PATH)]
for r in rows:
    k=r["kind"]; pay=r.get("payload",{}); prop=pay.get("proposal",{})
    if prop.get("encounter")=="ems-border" and k in ("block","permit","action"):
        reason=(pay.get("block") or pay.get("permit") or {}).get("reason","")
        j = prop.get("meta",{}).get("justification") if prop else None
        print(k, prop.get("type",""), "|", reason, "| has_justification:", bool(j))


block proposal.order.labs | Identity not bound EMS↔MRN (set meta.identity_override=True if true emergency) | has_justification: False
block proposal.order.ecg | Identity not bound EMS↔MRN (set meta.identity_override=True if true emergency) | has_justification: False


In [84]:
# Explicit ICU bed request → should BLOCK when ICU is full
ledger_init()
enc="ems-check-icu"; pat="tmp-check-icu"

msg = base_msg(
    "proposal.bed.request",
    pat, enc,
    {"service":"Trauma","level":"ICU"},
    {},  # no attending_override
    "coord.capacity"
)
BUS.publish("proposal.bed.request", msg)
time.sleep(0.1)

# Show only decisions for THIS encounter
rows = [json.loads(x) for x in open(LEDGER_PATH)]
for r in rows:
    k = r["kind"]; pay = r.get("payload",{})
    prop = pay.get("proposal",{})
    if prop.get("encounter")==enc and k in ("block","permit","action"):
        reason = (pay.get("block") or pay.get("permit") or {}).get("reason","")
        print(k, prop.get("type",""), "|", reason, pay.get("note",""))


block proposal.bed.request | ICU full — propose ED hold or transfer 


In [85]:
# Test attending override for pregnancy CT
ledger_init()

# Proposed CT in pregnancy WITHOUT override → block
BUS.publish("proposal.order.ct", {
  "id":"evt_ct_preg_block",
  "type":"proposal.order.ct",
  "patient":"tmp-p1","encounter":"ems-p1",
  "payload":{"study":"CT abdomen","contrast":"iodinated"},
  "meta":{"pregnancy_flag": True, "life_saving": False},
  "provenance":{"agent":"coord.diagnostics","policy":"0.3","ts":datetime.utcnow().isoformat()}
}); time.sleep(0.1)

# Now WITH attending override → permit
BUS.publish("proposal.order.ct", {
  "id":"evt_ct_preg_ok",
  "type":"proposal.order.ct",
  "patient":"tmp-p1","encounter":"ems-p1",
  "payload":{"study":"CT abdomen","contrast":"iodinated"},
  "meta":{"pregnancy_flag": True, "attending_override": True},  # override flips the rule
  "provenance":{"agent":"coord.diagnostics","policy":"0.3","ts":datetime.utcnow().isoformat()}
}); time.sleep(0.1)

for b in ledger_tail(10):
    if b["kind"] in ("block","permit"):
        payload = b["payload"].get("block") or b["payload"].get("permit")
        print(b["kind"], "→", payload.get("verdict"), "|", payload.get("reason"))


block → block | Identity not bound EMS↔MRN (set meta.identity_override=True if true emergency)
block → block | Identity not bound EMS↔MRN (set meta.identity_override=True if true emergency)


## Cell 9 — Inspect Ledger Tail

In [86]:

tail = ledger_tail(40)
print(f"Ledger entries (last {len(tail)}):")
for b in tail:
    print(b["ts"], b["kind"], "hash:", b["hash"][:8], "prev:", (b["prev"] or "")[:8])
    if b["kind"] in ("permit", "block"):
        payload = b["payload"].get("block") or b["payload"].get("permit")
        print("  verdict:", payload.get("verdict"), "| reason:", payload.get("reason"))


Ledger entries (last 7):
2025-08-09T18:03:16.489400 genesis hash: 7cec9255 prev: 
2025-08-09T18:03:16.489975 block hash: 260501d4 prev: 7cec9255
  verdict: block | reason: Identity not bound EMS↔MRN (set meta.identity_override=True if true emergency)
2025-08-09T18:03:16.490243 audit hash: 0e0b0d58 prev: 260501d4
2025-08-09T18:03:16.490855 audit hash: bc8546cc prev: 0e0b0d58
2025-08-09T18:03:16.591568 block hash: 38b3091c prev: bc8546cc
  verdict: block | reason: Identity not bound EMS↔MRN (set meta.identity_override=True if true emergency)
2025-08-09T18:03:16.591836 audit hash: 561e9485 prev: 38b3091c
2025-08-09T18:03:16.591914 audit hash: ecd0c837 prev: 561e9485


In [87]:
# Cell 9.5 — Helpers: attending override + identity bind

def attending_override(proposal_type: str, patient: str, encounter: str, payload: dict):
    msg = base_msg(proposal_type, patient, encounter, payload, {"attending_override": True}, "ed.attending")
    BUS.publish(proposal_type, msg)

def bind(encounter: str, mrn: str):
    bind_identity(encounter, mrn)
    print(f"Bound {encounter} -> MRN {mrn}")


In [88]:
ledger_init()
enc = "ems-id-1"; pat="tmp-id-1"
# attempt ECG order before binding -> block
BUS.publish("proposal.order.ecg", base_msg("proposal.order.ecg", pat, enc, {"repeat":"30min"}, {}, "coord.diagnostics"))
time.sleep(0.1)
print([ (b["kind"], (b["payload"].get("block") or {}).get("reason","")) for b in ledger_tail(3) ])

# bind identity, try again -> permit
bind(enc, "MRN123")
BUS.publish("proposal.order.ecg", base_msg("proposal.order.ecg", pat, enc, {"repeat":"30min"}, {}, "coord.diagnostics"))
time.sleep(0.1)
print([ b["kind"] for b in ledger_tail(3) ])


[('block', 'Identity not bound EMS↔MRN (set meta.identity_override=True if true emergency)'), ('audit', ''), ('audit', '')]
Bound ems-id-1 -> MRN MRN123
['permit', 'audit', 'audit']


In [89]:
# Hard reset: new bus, rebind exactly one instance of each agent
ledger_init()  # optional: start with a clean ledger

BUS = EventBus()  # NEW bus -> no old subscribers

# Rebind governor first (so it can see proposals)
GOVERNOR = SafetyGovernor(BUS, POLICY)

# Recreate agents on the new bus (use the *current* class versions you have)
RISK = PerceptionRiskAgent(BUS)
CRITIC = CriticAgent(BUS)
DIAG = DiagnosticsCoordinator(BUS)
CAP = CapacityCoordinator(BUS)
ACT = ActuatorAgent(BUS)        # or ExtendedActuator if that's your current class
FALLBACK = FallbackAgent(BUS)   # the dedupe+cooldown version
AUD = AuditAgent(BUS)

print("Mesh reset and rebound: single subscribers per role.")


Mesh reset and rebound: single subscribers per role.


In [90]:
# --- Test A: ICU available path ---
ledger_init()

# ICU has 1 free bed
CAPACITY["ICU"]["occupied"] = CAPACITY["ICU"]["total"] - 1
CAPACITY["ICU"]["jokers"] = 1  # irrelevant here but set for completeness

enc = "ems-border-A"; pat = "tmp-border-A"
bind(enc, "MRN-A")  # satisfy identity gate

# Non-STEMI medical patient with high risk markers (borderline ICU)
RISK.publish(pat, enc, "major_trauma", {  # using this scenario so stub raises icu_need via NEWS2; not actual trauma semantics
    "sbp": 95, "hr": 122, "rr": 26, "spo2": 90, "gcs": 13, "lactate": 3.2, "fio2": 0.4
})
time.sleep(0.2)

print("=== After proposals/permits ===")
rows = [json.loads(x) for x in open(LEDGER_PATH)]
for r in rows:
    k=r["kind"]; pay=r.get("payload",{}); prop=pay.get("proposal",{})
    if prop.get("encounter")==enc and k in ("block","permit","action"):
        reason=(pay.get("block") or pay.get("permit") or {}).get("reason","")
        just = prop.get("meta",{}).get("justification") if prop else None
        print(k, prop.get("type",""), "|", reason or pay.get("note",""), "| justification:", bool(just))

# Simulate 2h to complete physical ICU admit
advance_minutes(120)

print("=== After +2h (expect icu_admit_complete) ===")
for r in [json.loads(x) for x in open(LEDGER_PATH)][-12:]:
    k=r["kind"]; pay=r.get("payload",{}); prop=pay.get("proposal",{})
    if k in ("action","permit","block"):
        reason=(pay.get("block") or pay.get("permit") or {}).get("reason","")
        print(k, prop.get("type","") if prop else "", "|", reason or pay.get("note",""))


Bound ems-border-A -> MRN MRN-A
=== After proposals/permits ===
permit proposal.order.ct | All invariants satisfied | justification: False
permit proposal.bed.request | All invariants satisfied | justification: False
=== After +2h (expect icu_admit_complete) ===
permit proposal.order.ct | All invariants satisfied
permit proposal.bed.request | All invariants satisfied
